# Распознавание типа космического корабля (Zero-Shot Classification)
В этом ноутбуке мы используем модель CLIP для анализа скачанных фотографий ракет и предсказания их типа без предварительного обучения на нашем датасете.

In [1]:
import os
import glob
import json
import time
import socket
import platform
import psutil
import pandas as pd
import torch
from PIL import Image
from datetime import datetime
from transformers import CLIPProcessor, CLIPModel

In [2]:
# --- Задание 3: Метрики для удаленных воркеров ---
# Сбор информации о среде выполнения и производительности

def get_worker_metrics():
    """Сбор метрик о воркере, на котором выполняется ML анализ"""
    metrics = {
        "timestamp": datetime.now().isoformat(),
        "worker_hostname": socket.gethostname(),
        "worker_ip": socket.gethostbyname(socket.gethostname()),
        "platform": platform.platform(),
        "python_version": platform.python_version(),
        "cpu_count": psutil.cpu_count(),
        "cpu_usage_percent": psutil.cpu_percent(interval=1),
        "memory_total_gb": round(psutil.virtual_memory().total / (1024**3), 2),
        "memory_available_gb": round(psutil.virtual_memory().available / (1024**3), 2),
        "memory_usage_percent": psutil.virtual_memory().percent,
        "disk_usage_percent": psutil.disk_usage('/').percent,
        "gpu_available": torch.cuda.is_available(),
        "gpu_count": torch.cuda.device_count() if torch.cuda.is_available() else 0,
    }
    
    if torch.cuda.is_available():
        metrics["gpu_name"] = torch.cuda.get_device_name(0)
        metrics["gpu_memory_total_gb"] = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)
    
    return metrics

def log_processing_time(start_time, end_time, num_images):
    """Логирование времени обработки для метрик производительности"""
    duration = end_time - start_time
    metrics = {
        "processing_duration_seconds": duration,
        "images_processed": num_images,
        "average_time_per_image": duration / num_images if num_images > 0 else 0,
        "processing_rate_images_per_second": num_images / duration if duration > 0 else 0
    }
    return metrics

# Сохраняем метрики воркера
worker_metrics = get_worker_metrics()
print("=" * 60)
print("МЕТРИКИ УДАЛЕННОГО ВОРКЕРА")
print("=" * 60)
for key, value in worker_metrics.items():
    print(f"{key}: {value}")
print("=" * 60)

МЕТРИКИ УДАЛЕННОГО ВОРКЕРА
timestamp: 2026-04-03T09:56:36.953100
worker_hostname: 87df618284e8
worker_ip: 172.18.0.6
platform: Linux-6.8.0-101-generic-x86_64-with-glibc2.36
python_version: 3.11.8
cpu_count: 2
cpu_usage_percent: 10.4
memory_total_gb: 6.04
memory_available_gb: 1.72
memory_usage_percent: 71.5
disk_usage_percent: 67.7
gpu_available: False
gpu_count: 0


In [3]:
# Инициализация модели
model_id = "openai/clip-vit-base-patch32"
processor = CLIPProcessor.from_pretrained(model_id)
model = CLIPModel.from_pretrained(model_id)

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
# Возможные классы для распознавания
candidate_labels =[
    "SpaceX Falcon 9 rocket",
    "Soyuz rocket",
    "Space Launch System (SLS) rocket",
    "Starship spacecraft",
    "Ariane 5 rocket",
    "Electron rocket",
    "Atlas V rocket"
]

In [5]:
# Получаем список скачанных изображений из директории data/images
image_folder = "./data/images"
image_paths = glob.glob(os.path.join(image_folder, "*.jpg")) + glob.glob(os.path.join(image_folder, "*.jpeg"))
print(f"Найдено изображений: {len(image_paths)}")

results = []
processing_start = time.time()

# Анализ каждого фото
for img_path in image_paths:
    try:
        image = Image.open(img_path).convert("RGB")
        
        # Подготовка данных для модели
        inputs = processor(text=candidate_labels, images=image, return_tensors="pt", padding=True)
        
        # Получение предсказаний
        outputs = model(**inputs)
        logits_per_image = outputs.logits_per_image
        probs = logits_per_image.softmax(dim=1).detach().numpy()[0]
        
        # Находим наиболее вероятный класс
        best_idx = probs.argmax()
        predicted_label = candidate_labels[best_idx]
        confidence = probs[best_idx]
        
        results.append({
            "image_name": os.path.basename(img_path),
            "predicted_rocket": predicted_label,
            "confidence": round(confidence * 100, 2)
        })
        print(f"{os.path.basename(img_path)} -> {predicted_label} ({confidence:.2%})")
        
    except Exception as e:
        print(f"Ошибка при обработке {img_path}: {e}")

processing_end = time.time()

Найдено изображений: 3


model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

tianlong-3_full_image_20260330045730 (1).jpg -> Space Launch System (SLS) rocket (31.74%)
falcon2520925_image_20221009234147 (1).jpeg -> SpaceX Falcon 9 rocket (89.55%)
soyuz-5_assembl_image_20260327210910.jpeg -> Ariane 5 rocket (51.17%)


In [6]:
# --- Задание 3: Сохранение метрик производительности ---
processing_metrics = log_processing_time(processing_start, processing_end, len(image_paths))

print("\n" + "=" * 60)
print("МЕТРИКИ ПРОИЗВОДИТЕЛЬНОСТИ")
print("=" * 60)
for key, value in processing_metrics.items():
    print(f"{key}: {value}")
print("=" * 60)

# Сохранение результатов для использования в Streamlit
df = pd.DataFrame(results)
df.to_csv("./data/ml_predictions.csv", index=False)

# Сохранение метрик воркера и производительности в отдельный файл
worker_metrics.update(processing_metrics)
worker_metrics["model_used"] = model_id
worker_metrics["candidate_labels_count"] = len(candidate_labels)

with open("./data/worker_metrics.json", "w") as f:
    json.dump(worker_metrics, f, indent=2)

print(f"\nРезультаты сохранены в ./data/ml_predictions.csv")
print(f"Метрики воркера сохранены в ./data/worker_metrics.json")
df.head()


МЕТРИКИ ПРОИЗВОДИТЕЛЬНОСТИ
processing_duration_seconds: 1.355710506439209
images_processed: 3
average_time_per_image: 0.451903502146403
processing_rate_images_per_second: 2.212861806227008

Результаты сохранены в ./data/ml_predictions.csv
Метрики воркера сохранены в ./data/worker_metrics.json


,image_name,predicted_rocket,confidence
0,tianlong-3_full_image_20260330045730 (1).jpg,Space Launch System (SLS) rocket,31.740000
1,falcon2520925_image_20221009234147 (1).jpeg,SpaceX Falcon 9 rocket,89.550003
2,soyuz-5_assembl_image_20260327210910.jpeg,Ariane 5 rocket,51.169998
